In [50]:
# Ingest: set ROOT and data/raw/
from pathlib import Path

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Raw dir exists: {RAW_DIR.exists()}")


Project root: /Users/abnegusu/Desktop/Github/AI-Sec-Rag
Raw data dir: /Users/abnegusu/Desktop/Github/AI-Sec-Rag/data/raw
Raw dir exists: True


We found the project folder and created the empty **raw** folder where downloads go.


In [16]:
# Ingest: count and list files in raw/
raw_files = sorted(RAW_DIR.iterdir())

print(f"Found {len(raw_files)} file(s) in {RAW_DIR.name}/")
for path in raw_files:
    print(f"  - {path.name}")


Found 2 file(s) in raw/
  - owasp-a01-broken-access-control.md
  - owasp-top-ten.html


There are two files in **raw**: the good OWASP markdown doc and the HTML landing page.


## Ingest — inventory


In [17]:
# Ingest: inventory (name + bytes)
for path in sorted(RAW_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:45} {path.stat().st_size:,} bytes")


owasp-a01-broken-access-control.md            9,306 bytes
owasp-top-ten.html                            94,018 bytes


The markdown file is small (real content). The HTML file is big but mostly page shell, not good for RAG.


## Ingest — processed


In [18]:
# Ingest: copy A01 md → processed txt; sets `text`
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

source = RAW_DIR / "owasp-a01-broken-access-control.md"
text = source.read_text(encoding="utf-8")

dest = PROCESSED_DIR / "owasp-a01-broken-access-control.txt"
dest.write_text(text, encoding="utf-8")

print(f"Wrote {len(text):,} characters to {dest.relative_to(ROOT)}")


Wrote 9,304 characters to data/processed/owasp-a01-broken-access-control.txt


The A01 markdown was copied into **processed** as plain text. The variable `text` holds that full document in memory.


## Chunk


In [39]:
# Chunk: split `text` into 500-char pieces
CHUNK_SIZE = 500

chunks = []
start = 0
while start < len(text):
    end = start + CHUNK_SIZE
    chunks.append(text[start:end])
    start = end

print(f"Split into {len(chunks)} chunks")
print(f"First chunk preview: {chunks[0][:80]!r}")


Split into 19 chunks
First chunk preview: '# A01:2021 – Broken Access Control    ![icon](assets/TOP_10_Icons_Final_Broken_A'


The document was cut into **19 pieces** of about 500 characters each. The first piece starts at the A01 title.


In [ ]:
# Chunk: save chunks to data/chunks/*.jsonl
import json

CHUNKS_DIR = ROOT / "data" / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

out_path = CHUNKS_DIR / "owasp-a01-chunks.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks):
        record = {"source": dest.name, "chunk_index": i, "text": chunk}
        f.write(json.dumps(record) + "\n")

print(f"Wrote {len(chunks)} lines to {out_path.relative_to(ROOT)}")


Wrote 19 lines to data/chunks/owasp-a01-chunks.jsonl


All 19 pieces were saved to a file—one JSON line per chunk, with index and text.


## Embed


In [51]:
# Embed: load chunks from jsonl into `loaded`
import json

chunks_path = ROOT / "data" / "chunks" / "owasp-a01-chunks.jsonl"
loaded = []
with chunks_path.open(encoding="utf-8") as f:
    for line in f:
        loaded.append(json.loads(line))

print(len(loaded), loaded[0].keys())


19 dict_keys(['source', 'chunk_index', 'text'])


We loaded those 19 chunks back from disk. Each row has the filename, piece number, and the words.


In [1]:
# Embed: one chunk → one vector (384 numbers)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
vector = model.encode(loaded[0]["text"])

print(len(vector))
print(vector[:5])


384
[-0.01228833  0.11226239  0.00369145  0.04780543  0.0513843 ]


Chunk 0 became **384 numbers** (its embedding). Those decimals are just the first few numbers in that list.


In [1]:
# Embed: all chunk texts → matrix 19 × 384
texts = [record["text"] for record in loaded]
vectors = model.encode(texts)

print(len(vectors), vectors.shape)


19 (19, 384)


Every chunk got its own embedding: **19 rows**, each row **384 numbers**—ready to compare to a question later.


## Store


In [1]:
# Store: save vectors to data/embeddings/ (re-load without re-embedding)
import numpy as np

emb_dir = ROOT / "data" / "embeddings"
emb_dir.mkdir(parents=True, exist_ok=True)

vec_path = emb_dir / "owasp-a01-vectors.npy"
np.save(vec_path, vectors)

print(vec_path.exists(), vec_path.stat().st_size)


True 29312


The embeddings were saved to disk so you can reload them without running the model again.
